# Integração do modelo ajustado com contexto institucional

A composição utiliza Runnable do LangChain para transformar pergunta e evidências em um prompt e chamar o adaptador. O encadeamento mantém explícitas as entradas de cada etapa: a consulta fornece dados estruturados, a recuperação fornece trechos e a composição reúne essas informações antes da geração. A consulta ao SQLite é uma função Python controlada; a LLM não escolhe SQL nem escreve no prontuário.

O contexto de paciente é obtido a cada consulta. Na demonstração, os registros são estáticos e sintéticos; “atualizado” significa ler o estado atual da base, não conexão com um hospital real.

In [ ]:
from pathlib import Path
import json, os, sys
import warnings

warnings.filterwarnings("ignore", message="IProgress not found.*", module="tqdm.auto")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ["HF_HOME"] = str(ROOT / ".hf-cache")
import subprocess
subprocess.run([sys.executable, "scripts/init_database.py"], check=True)
from clinical_assistant.data_access import ClinicalRepository
patient = ClinicalRepository("data/processed/hospital.db").get_patient_context("PAC-0001")
print(patient.as_prompt_context())

Identificador institucional: PAC-0001
Ano de nascimento: 1968; sexo registrado: F
Condições registradas: hipertensão;diabetes tipo 2; alergias: penicilina
Exames pendentes: creatinina (solicitado em 2026-07-20); eletrocardiograma (solicitado em 2026-07-27)
Resultados recentes: hemoglobina glicada: 7.8% (2026-06-20)


## Recuperação lexical

TF-IDF representa o texto por pesos associados aos termos. Para cada termo, o peso é sua frequência relativa multiplicada por log((1+N)/(1+df))+1, em que N é o total de documentos e df é a quantidade de documentos que contêm o termo. Termos menos frequentes na coleção recebem maior peso relativo. A similaridade cosseno compara os vetores normalizados da pergunta e dos documentos para ordenar os candidatos.

A aplicação usa apenas a pergunta na busca. Os dois documentos mais próximos acima de 0,08 são candidatos. Esse limiar é uma escolha inicial, não uma probabilidade de correção. O trecho é a seção com maior interseção de termos, limitada a 700 caracteres; o método pode perder sinônimos e contexto.

A recuperação é lexical. Não foi implementado um banco de embeddings densos nem se atribui essa capacidade ao TF-IDF.

In [2]:
from clinical_assistant.retrieval import ProtocolRetriever
retriever = ProtocolRetriever("data/raw/protocols")
sources = retriever.retrieve("Exames de acompanhamento de diabetes", k=2, minimum_score=0.08)
for source in sources:
    print(source.source_id, source.version, source.score)
    print(source.excerpt)

PROTO-DIABETES 1.0 0.3279
# PROTO-DIABETES — Acompanhamento de diabetes em consulta

**Versão sintética:** 1.0 — **revisão:** 2026-02-20
PROTO-HIPERTENSAO 1.0 0.128
## Verificações de acompanhamento

- confirmar técnica e repetição da medida quando houver valor isolado discrepante;
- revisar medidas anteriores, adesão relatada, sintomas e medicamentos registrados;
- conferir função renal e eletrólitos quando esses exames constarem no plano assistencial;
- organizar fatores de risco e pendências para discussão clínica.


## Inferência real

A célula seguinte carrega o modelo base e o adaptador salvo. A resposta exibida é bruta e serve à avaliação. O quarto notebook aplica o controle de saída. Essa separação permite observar falhas gerativas que seriam ocultadas por uma resposta de bloqueio.

In [3]:
from clinical_assistant.llm import T5Generator
from clinical_assistant.chains import build_clinical_chain
generator = T5Generator("models/clinical-t5-lora")
chain = build_clinical_chain(generator)
draft = chain.invoke({"question": "Exames de acompanhamento de diabetes",
    "patient_context": patient.as_prompt_context(),
    "protocol_context": "\n\n".join(s.excerpt for s in sources)})
print("Saída bruta do adaptador:", draft)

Saída bruta do adaptador: Recommenda em acuerdo - confirmar técnica e repetiço da medida cuando hay valor isolado discrepante; - revisar medidas anteriores, adeso relatada, sintomas e medicamentos registrados; - conferir funcionarios renal e eletrólitos cuando es exames constarem no plano assistencial; - organizar fatores de risco e pendientes para discutir clnica. Resposta:


O modelo não recebeu autorização para determinar conduta. A presença de protocolos no prompt não assegura que a resposta os tenha utilizado corretamente. A retenção de respostas é parte do fluxo seguinte.